## Importing libraries 

In [37]:
import pandas as pd
import numpy as np
from pathlib import Path

import warnings 
warnings.filterwarnings("ignore")

In [38]:
PROJECT_ROOT = Path.cwd().parent
data_path = PROJECT_ROOT / "data" / "raw_data.csv"
data = pd.read_csv(data_path)
df = data.copy()
df.head()

,YEAR,QUARTER,MONTH,DAY_OF_WEEK,MKT_UNIQUE_CARRIER,ORIGIN_AIRPORT_ID,ORIGIN,DEST_AIRPORT_ID,DEST,CRS_DEP_TIME,...,DIVERTED,ACTUAL_ELAPSED_TIME,AIR_TIME,DISTANCE,DISTANCE_GROUP,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY
0,2025,1,1,1,AA,10135,ABE,11057,CLT,600,...,0.0,NaN,NaN,481.0,2,NaN,NaN,NaN,NaN,NaN
1,2025,1,1,1,AA,10135,ABE,11057,CLT,600,...,0.0,121.0,91.0,481.0,2,NaN,NaN,NaN,NaN,NaN
2,2025,1,1,1,AA,10135,ABE,11057,CLT,600,...,0.0,126.0,93.0,481.0,2,NaN,NaN,NaN,NaN,NaN
3,2025,1,1,1,AA,10135,ABE,11057,CLT,606,...,0.0,126.0,110.0,481.0,2,NaN,NaN,NaN,NaN,NaN
4,2025,1,1,1,AA,10135,ABE,11057,CLT,1219,...,0.0,132.0,98.0,481.0,2,0.0,0.0,14.0,0.0,11.0


## Data Understanding

In [39]:
# Basic data info
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 14399178 entries, 0 to 14399177
Data columns (total 24 columns):
 #   Column               Dtype  
---  ------               -----  
 0   YEAR                 int64  
 1   QUARTER              int64  
 2   MONTH                int64  
 3   DAY_OF_WEEK          int64  
 4   MKT_UNIQUE_CARRIER   str    
 5   ORIGIN_AIRPORT_ID    int64  
 6   ORIGIN               str    
 7   DEST_AIRPORT_ID      int64  
 8   DEST                 str    
 9   CRS_DEP_TIME         int64  
 10  DEP_TIME_BLK         str    
 11  TAXI_OUT             float64
 12  TAXI_IN              float64
 13  CANCELLED            float64
 14  DIVERTED             float64
 15  ACTUAL_ELAPSED_TIME  float64
 16  AIR_TIME             float64
 17  DISTANCE             float64
 18  DISTANCE_GROUP       int64  
 19  CARRIER_DELAY        float64
 20  WEATHER_DELAY        float64
 21  NAS_DELAY            float64
 22  SECURITY_DELAY       float64
 23  LATE_AIRCRAFT_DELAY  float64
dtypes: floa

In [40]:
# Checking the total null values count in the each columns 
df.isna().sum()

YEAR                          0
QUARTER                       0
MONTH                         0
DAY_OF_WEEK                   0
MKT_UNIQUE_CARRIER            0
ORIGIN_AIRPORT_ID             0
ORIGIN                        0
DEST_AIRPORT_ID               0
DEST                          0
CRS_DEP_TIME                  0
DEP_TIME_BLK                  0
TAXI_OUT                 207630
TAXI_IN                  212225
CANCELLED                     0
DIVERTED                      0
ACTUAL_ELAPSED_TIME      246782
AIR_TIME                 246782
DISTANCE                      0
DISTANCE_GROUP                0
CARRIER_DELAY          11398923
WEATHER_DELAY          11398923
NAS_DELAY              11398923
SECURITY_DELAY         11398923
LATE_AIRCRAFT_DELAY    11398923
dtype: int64

In [41]:
# Sort the data based on year and month
df=df.sort_values(by=['YEAR','MONTH'],ascending=True)

In [42]:
# if flight had a column where it was delayed or diverted and the air time was null removing those rows
mask = (
    df['AIR_TIME'].isna() &
    ((df['CANCELLED'] == 1) | (df['DIVERTED'] == 1))
)

df = df[~mask]

In [43]:
# Dropping null values
df = df.dropna(
    subset=['TAXI_IN', 'ACTUAL_ELAPSED_TIME', 'AIR_TIME']
)

In [44]:
# Filling the null values
delay_cols = [
    'CARRIER_DELAY',
    'WEATHER_DELAY',
    'NAS_DELAY',
    'SECURITY_DELAY',
    'LATE_AIRCRAFT_DELAY'
]

# rows where any delay exists
df[delay_cols]= df[delay_cols].fillna(0)


In [45]:
# Checking the total null values count in the each columns 
df.isna().sum()

YEAR                   0
QUARTER                0
MONTH                  0
DAY_OF_WEEK            0
MKT_UNIQUE_CARRIER     0
ORIGIN_AIRPORT_ID      0
ORIGIN                 0
DEST_AIRPORT_ID        0
DEST                   0
CRS_DEP_TIME           0
DEP_TIME_BLK           0
TAXI_OUT               0
TAXI_IN                0
CANCELLED              0
DIVERTED               0
ACTUAL_ELAPSED_TIME    0
AIR_TIME               0
DISTANCE               0
DISTANCE_GROUP         0
CARRIER_DELAY          0
WEATHER_DELAY          0
NAS_DELAY              0
SECURITY_DELAY         0
LATE_AIRCRAFT_DELAY    0
dtype: int64

In [46]:
# Check for duplicate rows
df.duplicated().sum()

np.int64(10660)

In [47]:
# Dropping the duplicated rows
df.drop_duplicates(inplace=True)

## Feature Engineering 

In [48]:
df['operational_risk'] = (
    (df['CARRIER_DELAY'] > 0) |
    (df['LATE_AIRCRAFT_DELAY'] > 0) |
    (df['NAS_DELAY'] > 0) |
    (df['SECURITY_DELAY'] > 0)|
    (df['WEATHER_DELAY'] > 0)
).astype(int)

In [49]:
# Checking for the class balance of target variable 
df['operational_risk'].value_counts(normalize=True)

operational_risk
0    0.787845
1    0.212155
Name: proportion, dtype: float64

In [50]:
# Checking the shape of the data year wise
train_df = df[df['YEAR'] == 2024]
test_df  = df[df['YEAR'] == 2025]
print(train_df.shape, test_df.shape)

(7182295, 25) (6959441, 25)


In [51]:
# Drawing a stratified sample from the 2024 dataset for model training

target_n = 200001

train_sample = (
    train_df
    .groupby('operational_risk', group_keys=False)
    .apply(lambda x: x.sample(
        n=int(len(x) / len(train_df) * target_n),
        random_state=42
    ))
    .reset_index(drop=True)
)


In [52]:
# Drawing a stratified sample from the 2025 dataset for model testing
target_n = 100001

test_sample = (
    test_df
    .groupby('operational_risk', group_keys=False)
    .apply(lambda x: x.sample(
        n=int(len(x) / len(test_df) * target_n),
        random_state=42
    ))
    .reset_index(drop=True)
)


In [53]:
print(train_sample.shape)
print(test_sample.shape)

(200000, 24)
(100000, 24)


In [ ]:
# Extracting the train and test samples into csv
data_dir = PROJECT_ROOT / "data"

train_sample.to_csv(data_dir / "train_sample_2024.csv", index=False)
test_sample.to_csv(data_dir / "test_sample_2025.csv", index=False)